In [0]:
import base64
import gzip
import hashlib
import io
import tarfile

dbutils.widgets.text("run_id", "")
dbutils.widgets.text("run_open_ts", "")
dbutils.widgets.text("silver_update_id", "")
RUN = dbutils.widgets.get("run_id").strip()
RUN_OPEN_TS = dbutils.widgets.get("run_open_ts").strip()
SILVER_UPDATE_ID = dbutils.widgets.get("silver_update_id").strip()
assert RUN.startswith("dq4_silver_") and RUN.replace("_", "").isalnum(), RUN
assert RUN_OPEN_TS and SILVER_UPDATE_ID
LANE = "dq4_results"
PAYLOAD = """H4sIAAAAAAAAA+19aXPbxpa2PvtX8EtKUsaWutFY6ZvU5Ca6uZ5y7Ly2cqemUikMRYIyy1wULrYzk/nvby8ABYCkAILE6W7mPFm0cTkNnuc5D3pd/D6+nieL1XgZL34fx/3ZIInnvWUSLz6Mhsu4Px5NR/3eOO49PMxG0+UkmS6v+APP9gHh8H1XfKWBR/JfBRh16Bn1HJc5AQsc54w4zPPpWYfs9S4NsVose/NO5yzpLWfTu92Pq/q7pXj15v3Nu9vOqze3bzv/HcaD5NN/Xy1G40/JPP69fzX4Pe5/SPofY5Uiz97fvL75/rZzPvjdjdNHOcTxSeh458875zwxuqUM6m7LoPPnzzqd7797f9P5z3/evOn0Z6vp8uLry863HdK5Fb85H/ZG4/POzWv+kPOH3mLBv3/zg3hW9tjnnYs0GPWbH169v331Rv44SC47/3j39qfN9sTLycOViH05T6aDrckdb2la5wt/4xx40O9uOl+uJrPp8kPn2286vcEglj8sLnirbi/OxTNfkPAFoeed7953fvju9oZH/IJ6l8VX6nS+e/PD+pX+1tn97EvR+GFvvEied9788vp1+n/+y0nv4eJ8vhon4hOYJ33eiBfsxXA1Hr9QL/v+n9+9u3kxG754mM8Gq/6yM0l6087fyJXzpTObd771vnQe5qPZ/AV10mcsPvTmiXzYS/Un9RTeUsr5ej2ZPZeXubP+FXmZPueb7A/XnfW7yZdczpa98cvOZLRYjKb3HRWn+tuiI545nS07/5PMZy+Go/E4GcgUETj/PJoOZp9j/oGJBuauzfoRy9nDuYho2u8t48+Li/OX/MfFeNRPLhaz+TLuzee9Py76M/66/WU8Hi2WF+uPQT3rQkTNX/1P/kT5Gcx5Ug0ueBqPeFvZpfgY3t++e/Xmx0v+SXSW81XCv1D+L1GfjIyDK8m9/BTWKXYu/3j+w/9zRYCr+Vxk2HI0SfhDJw8Xl89knopo1snM45Cv1/t0f/FIkb1TjccsmaQ+Fk6ey861fN0pT4zR8GLby//tgFd/3iHyKqlLJt9psZpcTOUv5Ye/u1n7v+9UtYj/RaZnLNKTv/z6ahavp3qb553pc5mYaVxv/3XzrnPx83fvbl/dvnr7pvP3/1KPky8rGyZf6LhKsnz2KCDLQwRE6Mayjm7wx//47u0vP4v2SW18lr67+EzW/JWv93gts98/U2+kiMDfh2tG5+079SFztfYef3j1XgrS5UvdBQ2xFxY1/d/6myGXY67g+5jASv/nkpL/cz1C0P9BAM7/lTPINBNYjg+dIDpBdILoBBs6wVpygnYQoRd7+79FfzZP9usCrPJ/PnHL/o/nLvo/CGjwfzKDgNzf4yOm/eRhGX9M/hBaWCgGj9VJVI4dMi8uxafeeJXE3M1wYyJLgpLO+Woajwbc8Gy9LKmUKgfEH7Prejxb6/g4+ZSMxUNlfOd5vV434nlWodCQoiFFQ/oXNqQnr3DokdtEbf83m0xW09Hyj7jP0y/u9ZejT/ynWkawsv/P90r+zxe/Qv8HAED/tz2DjOsG3B4m9gai+ULzhearaW/gHqqChgcBhvr+bzoY8Q9xuvfsvxr+j7Ly/D+HYv8fCCD9X5pB5jm+NDD0eOjx0OOhx2vs8Z7SEXR1CANR1/8NZ/NJA+snUeX/SFAe/3X439H/QQDO/4kMMs36iZjQ9aHrQ9eHrq+h69spIWj4EOajrv/7ML+P77lGPuy59kOgev1v2f/xL7j+AwRw/i+fQab5wHxs6AfRD6IfRD/Y0A9WSgn6QoQ5qOv/RpPevcjo5Etv737ABv7P9xj6PwjA+b98Bpnm//Kxof9D/4f+D/1fQ/9XKSXo/xDmoLb/mw7412YTAKvn//ll/8c87P8DAaD/W2eQce5vHRl6P/R+6P3Q+zX1fk8LCTo/hGmo6/8myTq1e4PJaC8X2GT/P4r+DwRw/q+cQaa5wHJ86AXRC6IXRC/Y0AvWkhN0hAi9aOD/ZvNBMm/b/zEc/wWBFv8nM8hg/yfjQ/+H/g/9H/q/w/3fbjlB/4fQi7r+76G3/DAbz+7/2N/+Vfs/Vt7/z3UZnv8GAjj/V8og0+xfKTx0f+j+0P2h+2vo/uqoCZo/hHbs7//myQMX05b7/3wH938BgQ7/pzLIXAOo4kMHiA4QHSA6wIMd4BNyghYQoRdN/J949DH9n0+dDf/HXPR/ENDj/8SL4QlwG1fk2VrL8Qw4NKVoStGUWn8G3LE0Dp1yO9jH/33u/cF1o9f/uOcegE3m//k4/w8EsP4vn0Em9v/l48P+P7RaaLXQah3Q/1cpJ+hqEHpR2//NZ/1ksJonUOe/Yf8fCAD9X5ZBxhm/LDB0fOj40PGh42vq+J7UEbR6CANR1//Nk2HCpXTc5Ay4av9X7v9j/F/0fxCA839ZBplm/7K40P2h+0P3h+6voft7UkbQ/CHMRG3/t+Rq2l+OPo2Wf0Cc/4bnf8AA0P/lMsg4D5iLDX0g+kD0gegDm/rAKilBL4gwB3X93+Ih6XPVanD6R8P+Pxz/BQGc/8syyDTvl8WFvg99H/o+9H0Nfd+TMoKeD2Em6vo/fjsjknx038L5b16w4f/4V/R/EIDzf48ZhCt/c9ei4Xo4NKZoTNGY/oWN6UmrG/plKNT1f597o6VYxSSkigvfcr7HKHD1+t/y+b8exfN/YQDn/zYzyLSewM0IsU8QrRdaL7ReDfsEawoKuh2ELlT4v2XyZRkPZv3VhKdvk7FfgSr/R4Ly/D/H8Xz0fxBo3f8VMsgQx1eICT0eejz0eOjx9vN41RKCrg5hPq6ur67//fVsev96NP3Y0ntI/+e6md8rfyXUD9bfy99T6pPgrPO6pXgKyPzffDZbPvW4qr9bipL/H4zuR8v4QS5mS6b95PFe9nGiS3825+Zu3o9X82ncT+ZTcb5R/yEeTR9WS7EOLh4N/uSfYBQFLo2iOI6ns2nCv1CXucNB07sIRBuA+Pwr7v8o80rnPzIWYP8/DI5+/1fOoO6ODOryDOryDOqqDOo+kUHdkoKU7iKn8XJAxC3kFXddX/MfR/w+g7uVC/WHf5MP8ORN5hVbPyK7H6i47Sy8xnP11OfZbZm6Z1APuc7+5gpnzENZ36nlBjjlRardbvHo1XS0FF+zS/B4D5R8eeA3N8kg5g8ZzuYT8ShyRYl8j/x9STqsW+/OpMbw8EFjw8UkOC8ODadX6jz1ovmh7W/2unDy6eLSieetLx0a1k2Y4f/8Tf/no/+DQFv1n7mEeoHve2j/jAbE51/t/0rn//AnE5z/CwJz/d86g/5a9m/dbHR/e7m/x+uG5m8PGOH/vC3+z0P/B4Ej1X/OYa4n/cGfrh9Q5keUoe+zAhCff6X/c8r+j/m4/gsGxvi/zQw6ad+32Vz0e0/5vS3XC33eEVBb/8sH+cXUcQMS+HE8WsXxOB46Yc91tlb5Kv0Xh30U9Z84FPUfBID6X86gbppB3UIG2a3ys8nsIWtXXtB/Ha1+ux7bLucbh3nuI+iFS1PQ7vTioHJrwBH0X5I3YQOXhttv8qr1n5b1n7g4/g8CI/Q/l0EnKv+o/rvVH8VfIw7Qf5/6IanTz1ep/xv9P8SjuP83CLTqv8qg0+rnUUKnWnaKXTpHqALpxcH+GwPQXP8ZocTzw/heeDfXuXN2uP8a+k/K578GHg1Q/yGgU//TDOo+ZtAJSH/aqLz036P3f7wuBdW/R+evGYfov8crOW3J/xP0/yDQq/8yg07Q/6ctQ/+/vRKoi4P+3wAcoP8BDVw3qDHBv1r/y+v/iI/9PzDQqv9pBp1iAUibhhVgawXIrg6WAN04RP8dEnqUVA8AN5j/w2sC6j8E9Oq/yqBTGwDO2oUjwFuUP700OARsBg7Uf5/VmADURP8Z+n8QaNd/nkEnqf+8Xaj/2/VfXBrUfzPQXP9dLtKR78aryWzM2eu4lGv30fSf4P6/INCp/2kGdQsZdAL6n7YrL/+iiaj/j5emIP/pxUH914CD9D9wo6Cl+T8M5/+DQLP+iww6rfk/aaNw/s+m8svrgvN/jMLh+j8R9GU+GwbDHQWgWv/L578F/DYR9R8CJuh/LoNOswBMsALsqgATLAFacYj+O65H6mzv3mj9F47/gkCv/ssMOsHpP2nLcPbP9jqgLg5O/jEAh+h/FFLmtdT/42D/Pwj06r/MoFPr/5GNwv6fDdVX1wX7f4zCAfpPaRiFfkvjv9j/AwOt+q8y6PTGf1W7cPx3SxVILw2O/xqCg/Q/8j3Slv/H/h8QaNZ/kUEn5v9Vo9D/byq/vC7o/43CIfrvOVHIcP6P1dCr/zKDTk3/ZaNQ/zf0X10X1H+jcJD++6E456Wd/Z8p6j8ENOu/yKBTW/6VNgtXf22rAfLK4OIvY3CI/nMNp368iInLegHdfcBblf7zH8r+n7rY/wMCvfovM6ibZdBJaL9sUl77F6j72VUp6P4CNd8AHKT/vufTeCVu3wlJ7nq7OoCazP/3cf9nEGjWf5FB3VwGnUYJEK0qjP5iD1DuwhTHfrELSCsO0f/Q9VkQT8TsDfEfDXqDqHec+f/i4aj/ANCr/zKDuqUMOokaIFtWWAMmJrnw/7AOZBenuA4suzxYC0BxiP5HHmNhS/M/Keo/CPTqv8ygE5z/KduF8z+3ab+6NDj/0xAcpP9B5Adtzf/H9V8g0Kz/IoNOUf9Fu1D/t+q/vDSo/4bgIP2PiOO15f9x/g8INOu/yKBT1H/RLtT/rfovLw3qvyE4UP9ZW/P/HQ/1HwLa9Z+d3Px/2Sic/79N+RnO/zcMB+i/Q0KXui3t/0lw/g8ItOq/yqBT2/9TtQr3/9yoAOmFwf0/DcIh+i+E24snk/hDzHOnF+5aAtBA//H8XyDo1X+ZQd1cBp2E/stWFef+XH9A/c8uTGneD780qP/acIj+M+K5rK3zv3D/BxDo1X+ZQafX/6/ahf3/22qAujTY/28IDtJ/jzhtzf8k6P9BoFn/RQadov6LdqH+b9V/eWlQ/w3BIfrvB9QNYkriiLO3d8f6d9Hx5v+g/weBXv2XGdQtZNBJ6L9sV17/Kfk6Qv1/vDQF/U8vDuq/Bhyi/xGhUWvzf3D9Fwj06r/MoBOb/6MahfN/NpRfXRec/2MUDtJ/X+z/3875j7j/Pww067/IoFM8/1G1LF8E8PzHb0oXB89/NAAH6D8jvuM5nLZ3d0M2PO7+n8zF/h8QaNV/lUHdLINOQfhVk/LC/xUqfnZVCor/FUq9AThA/92Q+K5D4sk4noymcc8L2dbt32r0/wdl/8+/Rf2HgFb9TzOoW8igU6gCacMKc0DH17yNWAvW16Y4DVRdHawI0DhQ/0OXxlM1fSNhid/zj6T/Lvb/wEC7/vMM6hYy6FT0nzcsr/9TnAFUvDYF/Z/iFCA9OEj/KQnCqPoAsCbzf/D8Lxho1n+ZQSd3AFjarrz84wlgxUuDR4CZgUP1PyLt6D9D/QeBfv2PyGnqf0RQ/3fof0RQ/w3BwfrvtaT/uP8zCAzQf+9E9d9D/d+l/x7qvyForv+LeT/+/ubdm5t38e3N+9vv3/5w8+f3737esh9cpf57Zf13Ahfn/4NAp/7zDOpuyaDT2Q9uRwNxa7hcWdh1jXCXOBAcWf/fbzsOvlL/XVLWf+Li/s8gME7/35/IcfDbG5eX/r/2wfA7rg8eEQ+L4+r/P/7+/Zb1wNX+n5b1H9d/AcE0/ecZdDLrgXe0D5cGP10FxCXCVcJAOK7+//jj7ZbhgGr9L8//ZCRA/w8C0/SfZ9AJDQfsaCEODFTVAHGRcIgAAMfV/19utq0Ga6D/1Ef9B4Fp+v/LzSmtBtveQlwWVqn//CLh+jAIHKb/t//4/s/vfryp2AOoev+f8vgv9X2c/wkC3fqfZtBp7QGUaxhuAbRF7bNrgzsAacfh+n/z86vj639AsP8fBCboP8+g09R/3jDU/x36L64N6r92HK7/XDpu3n3/VAmo1H/mlcd/HQ/PfwGBCfqvMug0S4BqG1aBHVUgvTxYCHThcP3/+ft3N08fAlbt/8vzf6iL53/BwAT9Fxl0SoeA5duVV348BKx4afAQMP04XP9/efPLP5/uAKrW//L4r0Nc3P8TBCbov8ig03T/omXo/XfUAHlx0PnrxVH0/6fj67+H639BYIj+/3Sy+v8T6v9u/f8J9V8zDtf/f/2dOvFUrNnp991ouGUH6Or9f1i5/8ehOP4LAhP0X2RQN5dBp6H9olV57Z/iWq/chSnu/owrvHShtv5/Gi35/xej+2nsUp4lYRRPRuMxz8Alf348SXg5SHwu3XcbBaB6/4eN+T/Ew/n/IADU/8cM6qYZ1N2RQXYXALXRmWphYar/Y2M7orGr+R+214PHj7TBVnDpFSrO899yjbAwtIhm+u85NHJrnPyrUKn/xC+f/yi2hED9B4Au/ZcZdFqdPqmqyZadYqfPYWKvLgt29xiGQ/S/Tf+P478w0Kv/p+z/N+oA+v/qkoD+HxgN9d+VB7+06f+x/wcE2vTflQe/nKL/d0tHv6D/f7ws6P8NwyH632r/P87/B4Fe/T9p/1+uA+j/q0sC+n9gNNJ/h0UuCeM7rokLsVfXapnEvYEXem6z87+45pfO/3IIzv8EgSb9VxnU3ZJBp6D9qnV57ZcNvVYN/UuLfnppCqJfuDio9qBopP+M0NAPsf/nBKBJ/1UGnWL/j2oZ9v+UdD+9LNj/YxgO0X/0//ZDr/6fqP/frAHo/5+oA+j/9aGZ/lPmRRTnf54AdOm/zKCT9P+yZej/y7qvLgv6f8NwiP7fzbkoflj7N8cb3EUbm7+d1dn/s3z+L/EcH/UfAnr1f2sGnWYVSJuK9wDba0Hp8mBFAMLV9dX1v7+eTe9fj6YfW3oPqf+um+l9+SuhvrP+Xv6eUl/s//+6pXgKyPR/Ppstn3pc1d8tRZP6L3b+WM2ncT+ZT7laDPoPXBofVkteJYZcTP4MHV7tA8+rf4eI0AWIz7/K/zG33P/rEg/9Hwj0+D+xFQjPoK7KoO7uDDqtHoLazcbug8KuMfWuGfYtNIAZ/o9u+j+K/g8CR6j/nMdcT/qDPwMaMs/zg9oLAxDaAfH5V/u/8vk/ruPg/C8QGOH/NjLoNBeGVLUaF4vUMICbVw0XkByAkv5PV/yijfrxvDe9f2rzV0qI4zAST9SpLf071ifervpeof8kcFl5/N+nuP8HCI6u/4UMemLz1zSDuoUM2tB3+SpCv6vVWj5U6HQawVqr3/zy+nX6/6fGbNKACiKcHthTVlX+xk5DTY2Xk4cr8SKfRrzduwm25Qqn6rttSCWLvSiFdU7Uacz/yCHc6VES+/zT8wJ2F22f/HnWiP8O8h8GGvkvM6hbyCC9/JcB5flPyde+JfxXsRf4n0bfDv8dNyCBH8ermH94CRu4NGw6/4/zn5T5z3zs/weBPv6rDOrmM0gr/VU8efr/uvrNDvanoRfYr4KvvPVpyn+f+ow6sdi0P3GdO2c3+2vwnzll/hMXz38AgT7+qwzqPmaQVvaraPLsv7eD+2ngBe7vcZzCAfx3HbdF/4/1HwRa+c8zyCD/rwKy0/+nscP5f58zOGTc/49jd+g7fv/I9T/A+38Q6OS/yKDuYwZpJr+IJk/+65UlzJeBF5gvQm+3/jvUjUjohvHHh148dBzfYbsFoJr/3ib/cfwXBNr4n2VQ9zGDdApAFk5eAXhkFijAOvKCBIjY60hAQ/4zzlr+T3vjf4wg/yGgjf9pBpkz/pcGZOX4XxY72PgfE6s2WdSe//ex/oNAI/9lBhni/9No7PP/WeDA/l+8rReE7dV/F/f/AIFW/vMMMqn+y4Asrf8qdsj67/quH8fx3d2QDemTU/sr+e+U9v93fN9H/oNAJ/9FBnWzDNJMfRFLnvpf2cF6GXaB9V/tN925Mf8dXrZbrP8M+/9BoJH/MoNMqv8yIEvrv4odsP47LIqYGP+P+KfX45/eXXRE/rsM938FgU7+iwzqFjJIM/9FQKXx/8gW/svYy+P/UYv8dwNGBf+pwz8+FlFnsGt/l2r+b+z/yVw8/wsEOvkvMqhbzCDNAiAiKgkAdWxRABl8WQFk+E9IwIH8b6//H+s/CLTz35T+/w3u29L/v4X3EP3/ovPBi+PRKo4n4/jOpz2ya4OPBvU/8HD+Pwg08l9mULeYQXo1QEZUWAE0Wv12PbFDB1TwxTVAafht1H9GXD9osf5j/x8ItPKfZ5Ax9V9GY2P9V4FrqP+e54n7f5YuAg6DO3bX2yYDlfz3gvL+7zTA+X8g0Mp/nkHdjQzSLQM8qFIXALuyZSVwFn+5FyBrwaYoNOc/DcX8v/7sYZQspIPzWMSYv//5DyTwN85/cxmO/4NAJ/9FBnU3M0izAIio8gLwvyrA/7PmPkA2oKAA+SaUJOBA/o/vHyVAfIIR85y78kLgSv4Hpft/RhnD/n8QaOf/9gwyTAPG9xcZhy4t1oFyM14ewn/Xdfx4EfNS3QuOPv/P9fD+HwQ6+S8yqJtlkGbGi1jyjF/YwXEZdoHjC6D5f8x3w6g9/gfY/w8CnfwXGWQK/0UsFvJfhq2H/1FI3Bbn/yH/QaCT/yKDTJr/JwOydP6fih1w/p9HvMhpcf8f7P8HgUb+ywwyZvxPRmPj+J8KHHz8z3OI3+L+Xz72/4FAJ/9FBpnDfxGNlfyXgUPz33eI67Cw1gBg9fhf+fwn4lHc/wME+vifZpBpA4BpWBaPAGYtqDUE2JT/AeUUdf1aC4CbzP/F+38Y6ON/mkEGLQBOI7JzBXAW/J5LgJvz3w+jMKx1AEiD/j+P4P0/CHTyX2aQMQeAZAEV5v/bMu8vi33vI0Ca858zNfTiqTi/od93o+G2mX8K1fzfOP/HJbj/Pwh08l9mUDeXQZrpL+PJ039qxxkg69AL7J/WOgWkMf8dEnqUtFX/XQ/rPwg08l9lkEH1XwVkZ/1PY4er/+INvai1+3/c/xMGmvnvRSbd/6uILL3/T4OHuv/nb+c7tDX+Ozj/FwR6+c8zyDD+84js5b8IHpL/rN4BwE38P+7/CwPN/GfmHACcBWSt/xexg/p/P2TxSqm344ouiOPVfw/X/8JAM/9D1i1kkH4BKB4CuLKr/pcPAly1Wv9DRlvjv4P7/4FAL/95BhnG/7C4CaBd/A/LGwG2y/+ItNj/h/3/INDLf55Bht3/84jsvf8XwQPe/0c0iqfq00tY4vd2zQBown8P6z8INPOfRt1CBunnPy3wf2oX/2mJ/9P2+M8ixyVhrQPAKvnvBOX1/4GP9R8EGvmvMsiMA8DSYPLUt+IEsCzuxkeAHcB/5hMvnojZW8xnw2B4zPl/zMH5fyDQyn+RQd1cBumWABFPwf3bMv8vDb3o/dud/8ciz/OpmP8rNm0b+CzcuQS4yfx/iv4fBFr5LzKom88g3QIgAipNALZjzV8We3kG8NOb/58dxP8gpE6tDcAr+b+5/6+D+//DQCv/RQYZtgF4GlVpFyCLdgDPGlBzC/AD+B+6NGrN/zP0/yDQyn+RQSb5fxGPpf5fhg7n/11CPI+EtehfY/8fr8x/B8//hYE2/qcZZAr903AsZH8WeQPynx3C/8D3nRb5j/4fBDr5LzLIIP6LcOzkv4wcmP+UuOr2f3335rLeMNk2B7DB/T+jOP4HAo38lxnU3cwgvTIgoyrd/r+w5/4/a0D59v/Fce//+ds4oeO3t/83jv/BQCf/RQaZs/93GlCJ+nbs/53FDrb/d8Z/LgBcvz/f9eM4pNSnxzr/k1DkPwi0838jgwzTgGse39X/8vj+z1YdyLXgaOd/8jdike/WWv7T7PwP7P8DgU7+iwwyZ/lPGpCVq3+y2Pdc/HN2CP/dwA9bO//bIbj+FwQ6+S8yyKzh/zQoa0f/s/hbP/+bv5HvsCgejmMvYkNncOz5/w7yHwQ6+S8yqLvOIM28F8HkeT+0hO8y7gLfh7UP/zg7hP8B87z4Qbm3Yc/t9dgR/X/g4Pk/INDJf5FB3UIGadYAEVBeAx4s8v8y9oIOPLTq/7lDJ0Gt5b9N+O/j+n8YaOS/zCBzlv+mAVm5+jeLfc/Fv2cH8J8xEjm1tv9oxH/s/4OBRv7LDDJn+480ICt3/8hi33Pzj7MD+O86XsDa4z/Wfxho5L/MIJP4LwOylP8qdkj+h5zUQSxmbyauc+fs3P230fxfiut/YKCT/zKDuo8ZpJn9Mpw8/W2Z/5tGXiB/6/N/3ZDx0Nrb/x/P/wWBVv6LDDLKAMiIbHUAKniY/f/E27le4Mare24BQneQkKOe/8eQ/zDQyn+RQd1cBulmv4inMP/n/vreEu7L0IvTf0Tw7a3/DbyIEKet8z9ZgP4fBBr5rzLIlPM/s3hK2//YUfvT0OHO/3TFmkMxANjW/j8e8h8EOvkvM8i0CYAqKotnAKYNaHf/H/U2/KNrjf/Y/w8CzfynxET+Fw8BtpD/5ZOAW+B/yENx2pv/g/yHgUb+ywwyaf6PDMjS+T8qdsD5P5HrRV4cj6RwD52w5x5z/Z/n4fx/EGjkv8ygbiGD9PJfBpTn/68ja6q+ir14/OeovfM/1/yvcfxvs/4/3P8LBPr5b8jxv9vobzX7a5D/7BD+h5R5tab/1Jj/45T5T/D8Lxjo5L/IIGOm/8hobJz9owJvNPnn7BD+R17U4v1/gPf/INDJf5FBJt3/y4Asvf9XscPd/1Ma+R5psf4j/0Ggj/8qg0yp/yoaC+t/Gjh0/aeM+X7YXv33sf8PBBr5LzPIoPqvArKz/qexA9Z/5oW0xfrv4v0/CHTyX2SQMfVfRmNj/VeBw9d/P3CiFtf/Yv8/CHTyX2SQQct/VEB2rv5JY4db/0tdTmHa4v5f6P9BoJH/MoMM2v9LBZTnvz37f6Wxw+3/Rd3Qp15753/g/j8w0Ml/kUEGnf+hAirN/LXk/I80drjzP6gbOSFpb/4Pnv8NA538FxlkzvwfFY+V83/S0CHn/1CPBBFpcf4/1n8QaOS/zCCT+v9lQJb2/6vYAfv/PdHxGIuPbhAGSbLz5v+sUf8/xf4/GOjkv8ig7mMGaSa/iCZPfmuYLwMvML+a9ms0578ThazF+T+4/h8EOvkvMsiY8T8ZjY3jfypw8PE/j7OVtrj/N/IfBDr5LzLIpPE/GZCl438qdsDxP/GGbe7/j/4fBJr5b9L+/yogi/kPuf8/9Vzmuu2t/8f6DwOd/BcZZND6fxWQnev/09jh1v9n/G9t/A/7/0Ggn//GjP9t0t9q9rc9/ue6Ub3t/5vN/3WQ/xDQyn+eQUb5fxGQrf5fxg7q/33mx3F8dzdkQ/rU8d81zv92y+d/cyD/IaCV/zyDulkG6aY+jyVP/a8sYb0Iu8D6r/Y4/PvsEP77IWU4/8926OS/yCCT/L+Ix1L/L0MH9f8+Y0GI4/+2QyP/ZQYZM/4vo7Fx/F8FDj7+77uUtbn+H9f/gUAn/0UGmcN/EY2V/JeBg/M/cHw3aHH9H9Z/EGjkv8wgk9b/yYAsXf+nYgdc/8cJTP14EROX9YKj9/+5uP8XDHTyX2RQN8sgzdQXseSpv7CD9TLsAusXQP1/ge/5VBz/O44JSe56T9wAVPt/b6P/j+L5vyDQyX+RQd1cBmmWABFOXgJWttwBqMjLh//WuwVozv8wCL14pUZvHWFAjjn/z8Xzf2Cgk/8ig7qFDNKsACKgggLYM/6vYi9qQKvj/6ETBjR+uI8HIRn0yVM3ANX+P9jo/yM4/wcEGvkvM6i7ziC93JfB5Ln/cG8F71XcxX2/7ve4BWjOf9dnQXvzfwLkPwh08l9kkCHzf2Qs9s3/UWFrmf+j+C9nb4r/aNAbRNsO/z6rwX9u9svnf1PkPwi087+UQYbJgJxDy/+zVQ3W8e8Sheb8jzxK46noven33Wi4c/evZv1/Dq7/AYFO/osM6uYySDP3RTh57k9t6f9TkRf3/mq9/y9yIl7/W+M/w/oPAo38lxlkDv9lOFbyX0UOzX9GKGtx/w+s/yDQyX+RQSat/5MBWbr+T8UOuP4vcnmNbnH+H47/gUAn/0UGmTT/TwZk6fw/FTvg/L/I5/+0d/6Pi/N/QaCT/yKDTDr/RwZUGAO0qP7L2AHP/4kCn8jzf3z+6XkBu4vcI/LfcZD/INDJf5FB3UIGaea/CKhU/31b+C9jL9d/v13+t7j/B47/w0Az/03Z/0PGYt/4vwpby/h/FIQtzv8JCEH+Q0Ar/0Nj5v/IWGzkf6hr/k8URJQJ/59tAhQGd+xu6wyg6vk/wcb8H4Lrf0Cglf88g7obGaRbCKLiGAB30ezKnp2AVPzl+4CsBZvS0Jz/EWEM9/+zHTr5LzLIlPovYrGw/suw9dR/yf/7eMDbk9wxf+foX7P5PwT3/wOBdv7nMsgwCbi/HlhS8zdVQMbe7vyflP9t7f/HkP8QMID/Zuz/tYX+FrO/9f2/HBK4YYvz/1zkPwj08V9lkEHz/1RAds7/S2OHm//nkDBiYXv8x/0/YKCT/yKDjOK/CMhW/svYIfkfOV6L+38R7P8DgU7+iwwyZv8vFU5h9x9L7gDSyGH3/xJv6pP25v/h/n8w0Mt/nxg0/08FZOf8vzR2uPl/DiVBm/P/POQ/CDTyX2aQGeN/Khbrxv/SsHWM//G3Dr2wxfF/5D8ItPKfZ5Ax/Oex2Mh/EbYe/nsBYS3e/yP/QaCT/yKDzLn/l+FYef+vIge+/6c+icIW1/8j/0Ggk/8igwxa/68CsnP9fxo73Pp/R+w87LQ4/o/r/0Cgk/8ig0wa/5MBWTr+p2IHHP/jDt1l7Z3/gev/YaCR/zKDDDr/QwVUuAOwh/8qdrjzPxzmMBrG+dPbd3YAVPOfbNz/+1j/QaCT/yKDuvkM0kx/EU+e/tas+ktDL7B/14K/Eprz3yNO2F79x/3/YKCT/yKDjKr/IiBb67+MHbL+8zeM4onovWU+GwbDo/b/OwzX/4FAM/+jbi6D9LO/sPvfxJb+fxV58d6/9f5/FlDXV/1/H+9jEkaO398hAdX139/o//OR/yDQyX+RQd1iBmmWABHRRgfgRytOAkyD3+wB/PjkcYCN+e+6gdPm/H/kPwg08l9mkDnj/zIcK8f/VeTA4/+u50RBe/x3cP8/EOjkv8ggg/gvwrGT/zJyaP77YRCJ+T/r7dtc1hsm2zoBK/nvO+X9/5iL5/+AQCf/RQZ1NzNIswyIqEqTgF5YtANg2oDyTKAXO7YAbMx/z/V9hvP/bIdG/ssMMmn+nwzI0vl/KnbA+X8+80LaIv+x/oNAI/9lBpnEfxmQpfxXsUPy3yNeEE/V6G3CEr+38wTgJvN/Pez/A4FO/osM6hYySDP/RUB5/k/tGf9XsRfPAG51/N8PqBu0WP9x/i8IdPJfZJBR9V8EZGv9l7ED1v+ABL7f4vofrP8g0Mh/mUEmrf+RAVm6/kfFDrj+J3AIc1oc/8f9f0Ggk/8ig8wZ/5PhWDn+pyIHHv8LQl/s/zOSAzdDJ+y5x5z/7+H+XzDQyX+RQd1CBmlWABFQYQHQyJpRPxV7cQXQqMYSoMb8D8XeQ7j/l+3QyH+ZQYbs/yVjyVPfjv2/VNha9v8KfT9w21v/g+f/wEAn/0UGmbP+R4Zj5fofFTnw+p8wYGF76/9dvP+HgU7+iwwyaP2/jMfO9f8qdND1/xGhUZvnf+L4Hwg08l9mkDHnf6pw8vS35fzPNHLY8z/X/G/r/E/c/wcEBvDfiPM/t9HfYva3f/5n5Ij9vx/U6O2w5/Z67Ij9/4GD/X8g0Ml/kUHdQgZplgCntAP4gz3j/yr2ggo8tDr+H/meS9rb/y/A/b9AoJP/IoMMuv+X8dh5/69Ch73/D13W5vofvP8HgU7+iwwyaf6vDMjS+b8qdsD5v4r/8Wocu0N/995fAk3u/wOc/wsC7fx/zCDDyH+9spb5IvSW7/8Z8R3PafH8T+Q/CPTxX2WQGfN/VCzWzf9Jw9Yx/4dR4gbS/2ebt4TBHbvrbROCSv5zspf2/6G4/ycMNPJfZlB3I4P0CoEMqnQPYNHuP2n85fuAHZv/nB3Af4+FvP7fx6L7lrjugO6+A2g0/wf3/wOBRv7LDOrmM0gv9WU8xRFAHpoVrFehl8YARfCt9f+5QRgRx621AUg1/zf2/w5w/i8M9PE/zSCDNgBJI7JzB5As+D23ADmE/45Dam0A0IT/2P8PA7385xlk0AYAaUR27gCQBb/nFgCH8J/fbLRX/3H8HwR6+c8zyLD6zyOyt/6L4KHqf0h8V9T/cTwZTeOeF7Joa+ffWbPzf3D9Pww08l9lULeQQXr5ryIq1P/xNQ/OCv6nwRfrvwq/Jf6HLm2v/mP/Hwj08p9nkEn1X0Vkaf1Pg4er/4HjyQ1Aff7peQG7i9zj8Z9RvP8HgVb+iwzqFjJIN/9FRKXRP98a/svgy0N/fnv8FweA1FkAWD3/b2P9L8X9P2Ggl//iABBDFgCm4Vi4AjCLvNESwOb8pySIvFoLgJrM//fx/A8Q6OS/zCBzFgClAVm5AiiLfe8lQPX4/2m05P9fjO6nsevQMGQub9b9PEkWcT8ZL0arRTz0+kOWbJWASv5vOf/Px/F/EEDx/zGDumkGdbdnkN5VADKyvAKkQXbSICG0IEe2fdcCqPALIlBuQKPz/wr8972AhK3zH/v/QaCF/zKDjOS/jMxe/qvwD+D/gkc+mib8m4fRx5wArL9Z9Gfz7SzfjQr+Oy4pzf8Xv0H/D4Kj87+YQd0dGVRi+WI1ufgwWsbTzr915NfkPv3uf5L57LIO+/uz3jhZ9JOLitd63iH8P/GgtUxI9u5WivlqLOk+5c1adBYfevPkW3LFc/hlR7z24y8I7Xz+kMyTjJydB/rtN+RlR7xv/nmbj+IP6o3Hnd6yM/32GypIIS+QgHrbeD77LMSHX7LbxxZedr5733l/++7Vmx8vHx+f3G9/dHK/9fEiuq1PUJd+yzOG4979fTKI0wYUnii/2X355SfJX5Ln2+XW1y6K63w2HPGLX0teL9LkfFT/9DWL5ST9Y+fbbzryQkux5L+WH+51/gEd+WGtX0TiNn0R+WiZhUQkoGiJbOwe7/n7r85v4jek+A5ZPPyybUbDcyeLgP99y/sn93tHsCMA+WltuR5ZAPLvmxGIX6evt73oCU351BuvkuzTTR+tytp8NY1HA1m3tkiMDI0/bbDqy9q2S1zk48bJp2QsH6XS9PxZ6Qo/tutyryVzJ4Wa9b/cP7GXA6is/7Tc/+fyn7D+QwCs/pczCB0AOgB0AOgAjuQANuQFPUAt1Kz/j30Se977C1TVfxaU7//5dzj/DwRg9f8xg7DyY+XHyo+V/0iVPycsWPP3Rrn+j5YJ/4jmox6/dkfq/q+s/w4t7//BfJz/A4Pj1/98BtXs/u/PVtPlxdc1a7167PO16Kvf/PDq/e2rN/LH9cjY5Z461DmCHD3ZFTlMksF5TcvxkMxfiMd3JsmAX8wOb8Kot+Sl51uHPPA3Hc5nk86MByXcg3rMy454wmKLkVjOHkT95Jemt4w/Ly7OX/IfF+OR8Eyz+TLuzee9Py76s/E46S/j8WixvFhfE/Wsi9x15dH9KZ4/W825Soj3vOSfx3K+SvgXyv8ll0ep6vwx6Wc8vCq8/fBKDG+Kr7kYntUuO50hf+x/vH31ps5j+7Ihb3mWXq1TYph9Kz7d/tVjHgyz75+l9aZ/tVGP0t8XR3ALDUwfkQ7hqsY+y/JyeFUjM4dXdXNzeFXMzmdZpRxe7bAO/avff3V/67x633nz9lbmbv7Xf8tMhfhd725xMVS/f5E+4JIbC/Hr7CdhLRwsxLpQr/4f1P1fo/6X1//wb3H+Dwig6n9F9/8pOYCKrkj0AOgBNHqA7dmJLuAvinr1/6Du/8r6T4Py/t+M4Px/GEDV/53d/6dU+Xd2RWLNx5qvseaX8xKrPSJFqf6LjIsnoy/L1fxo3f/V9Z+V5/8xn+L4PwiOXv/zGYTd/6Xh1e0DlJdPrVPKbEE2Y+DzaPmBa7MjF/gsOkmvL36kwhnIaQbPMxfwsrNYLR54We98oURMO5iqNUHKMiw6895yNOuMpp1fw+fU+e3RM6RPi7/IrRCLo/vipZ4awm/gN/IfWLtmIm8lHj+YSe/LRVakpqvxeDS8mIym6ldiwkjn7ze3/3nDkzKUnx51RLv5dahvO56BzW5fL/ziFfk8jqezaRLH5+WBb1XY+RN+fPf2l587f/+v/JXhv/7nd//iH2qOlt90nGfr9BUX53H6jJzFcEVo5+vSvBos71agVv0/rPu/uv67tNz/z//B+g8BoPr/V+7+Rw+AHuBgD7DP/HZ0AYi6qFX/D+v+r67/dGP+P8Hzv2AAVP//at3/WPOx5h9c8+vNbMdqj2iOzfovx26XSTxbLccjnqeHjwJU1H9K/fL+v24Q4P0/CNqo/6UM2mMPoKlYV7ZIprwU1XIE5edsLO57osBOEn5HNUhLrHq+Mxuy7nIl6gRbDJ4/8Nx88RBF0XHG0/NrstZv+djKncPtashbfJtzFbIq51/nsShfiquTNV3VVG9nST3+9mXFD3nbFmZYJczBHvrfuBe4Uv+Dcv+v5zqo/yAA1f8ae8BgBbC/AtTZzBprgBnYQ/8b9wJW6r+34f+ph+d/gABU/5/cAwSV337lf3rrYtR8BAKBQCAQCARCL/4/UTeMmgAoBQA="""

def qs(value):
    if value is None:
        return "NULL"
    return "'" + str(value).replace("'", "''") + "'"

def adapt(value):
    return (value
        .replace("dq4_silver_20260825", RUN)
        .replace("2026-08-25T10:50:49.897Z", RUN_OPEN_TS)
        .replace("f5c7c7ab-e37d-4a31-b9c2-b7631becb16a", SILVER_UPDATE_ID))

def unpack():
    archive = tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz")
    items = []
    for member in archive.getmembers():
        if member.isfile() and member.name.endswith(".sql"):
            items.append((member.name, adapt(archive.extractfile(member).read().decode("utf-8"))))
    return sorted(items, key=lambda x: x[0])

def execute(seq, name, sql):
    sha = hashlib.sha256(sql.encode()).hexdigest()
    spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
      (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
      VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'attempted',NULL,NULL,current_timestamp(),NULL,'DQ4')""")
    try:
        spark.sql(sql).collect()
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'ok',NULL,NULL,current_timestamp(),current_timestamp(),'DQ4')""")
    except Exception as exc:
        msg = str(exc)[:4000]
        spark.sql(f"""INSERT INTO 8_dev.silver_qc.dq_exec_log
          (run_id,lane,stmt_file,seq,stmt_sha256,status,error_text,rows_affected,attempted_at,settled_at,created_by_session)
          VALUES ({qs(RUN)},{qs(LANE)},{qs(name)},{seq},{qs(sha)},'error',{qs(msg)},NULL,current_timestamp(),current_timestamp(),'DQ4')""")
        raise

run_row = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_run WHERE run_id={qs(RUN)} AND finished_at IS NULL").first().n
assert run_row == 1, "run_id must identify one open dq_run row"

existing = spark.sql(f"SELECT count(*) n FROM 8_dev.silver_qc.dq_check_result WHERE run_id={qs(RUN)} AND created_by_session='DQ4'").first().n
assert existing == 15, f"expected 15 singles results before derived results, found {existing}"

items = unpack()
for seq, (name, sql) in enumerate(items):
    execute(seq, name, sql)

total = spark.sql(f"SELECT count(*) n, count(DISTINCT check_id) d FROM 8_dev.silver_qc.dq_check_result WHERE run_id={qs(RUN)} AND created_by_session='DQ4'").first()
assert total.n == total.d, total

print({"run_id": RUN, "lane": LANE, "statements": len(items), "status": "ok"})